
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.5
## Explicit GVH Auxiliary Residual and Complete Constraint-Ideal Reduction

**Auteur :** Charlemagne O Laurince  
**Branche :** `0.2C1_prediction_foundations`  
**Prédécesseur direct :** `0.3.2.7.3.7.3.3.1.4`  
**Traceabilité :** `ACTUAL-UPSTREAM-GVH-CONSTRAINTS / NON-FABRICATION`

---

# Mission

Cette étape doit reprendre **les contraintes auxiliaires GVH réellement dérivées en amont**, et non une base symbolique générique inventée.

La chaîne amont `0.3.2.7.3.6.1` fournit, sur la branche générique :

\[
c_{14}\neq0,\qquad c_{\rm time}\neq0,\qquad \Delta\neq0,
\]

la chaîne de contrainte de norme :

\[
\boxed{
p_\lambda\rightarrow\chi\rightarrow\psi\rightarrow\rho
}
\]

avec :

\[
\Phi_A^{GVH}
=
(p_\lambda,\chi,\psi,\rho).
\]

Le même maillon établit :

\[
\boxed{
\det C_4=\Delta^4,\qquad \operatorname{rank}C_4=4
}
\]

sur la branche générique. Les quatre contraintes auxiliaires sont donc **de seconde classe** dans cette branche locale.

---

# Correction logique importante

Le résidu auxiliaire concret du crochet HH,

\[
R_{\rm aux}^{GVH}[N,M],
\]

n'est pas un objet indépendant déjà publié en amont : il doit être **extrait du véritable calcul HH**.

Il serait donc circulaire de fabriquer aujourd'hui un polynôme arbitraire et de l'appeler
\(R_{\rm aux}^{GVH}\).

Cette étape fera donc trois choses honnêtes :

1. reconstruire la vraie base \(\{\Phi_A^{GVH}\}\) ;
2. construire deux moteurs exacts de réduction :
   - idéal polynomial,
   - réduction localisée sur la branche générique ;
3. construire explicitement le noyau de Dirac associé aux quatre contraintes de seconde classe.

Ensuite `.1.6` devra :

\[
\{H[N],H[M]\}_{\rm can}
\longrightarrow
R_{HH}^{\rm can}
\]

puis appliquer immédiatement les moteurs de réduction préparés ici.

Ainsi :

\[
\boxed{
R_{\rm aux}^{GVH}
\text{ sera un OUTPUT de .1.6, pas une INPUT fabriquée de .1.5.}
}
\]

Aucune fermeture HH n'est déclarée ici.

\[
\boxed{
R_{HH}=\texttt{BLOCKED-PENDING-.1.6}
}
\]

\[
\boxed{
\mathrm{DISPERSION\_READY=False}
}
\]


In [1]:

from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3.3.1.5")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.3.3.1.5
Python: 3.12.13
SymPy: 1.14.0



## 1 — Sources amont réellement utilisées

Cette étape importe seulement les objets déjà dérivés dans la chaîne :

### `0.3.2.7.3.6.1`

\[
H_{\rm kin}
=
-\frac{p_s^2}{4c_{\rm time}}
+
\frac{p_ip_i}{4c_{14}},
\]

\[
\chi=-s^2+v^2+1,
\]

\[
\psi
=
\frac{s p_s}{c_{\rm time}}
+
\frac{v^ip_i}{c_{14}},
\]

\[
\Delta
=
2\left(
\frac{v^2}{c_{14}}
-
\frac{s^2}{c_{\rm time}}
\right),
\]

\[
\mathcal A
=
-\frac{p_s^2}{2c_{\rm time}^2}
+
\frac{p_ip_i}{2c_{14}^2},
\]

\[
\boxed{
\rho=\mathcal A+\lambda_{\rm mult}\Delta.
}
\]

### `0.3.2.7.3.7.2.4`

Les variables canoniques full-field actives dans les crochets de hypersurface sont :

\[
(h_{ij},\pi^{ij}),\qquad
(s,p_s),\qquad
(v_i,p_v^i).
\]

### `.1.4`

Le bloc divergence a été fermé avec :

\[
B^i
=
-\frac{v^ip_s+s\,p_v^i}{\sqrt h}.
\]

La présente étape ne réécrit pas ce calcul.



## 2 — Reconstruction exacte de la chaîne auxiliaire GVH


In [2]:

c14, ct = sp.symbols(
    "c14 c_time",
    nonzero=True,
    real=True
)

s,v1,v2,v3 = sp.symbols(
    "s v1 v2 v3",
    real=True
)

ps,p1,p2,p3 = sp.symbols(
    "p_s p1 p2 p3",
    real=True
)

lam,plam = sp.symbols(
    "lambda_mult p_lambda",
    real=True
)

vnorm2 = v1**2 + v2**2 + v3**2
pnorm2 = p1**2 + p2**2 + p3**2
vp = v1*p1 + v2*p2 + v3*p3

chi = sp.expand(
    -s**2 + vnorm2 + 1
)

Hkin = sp.expand(
    -ps**2/(4*ct)
    + pnorm2/(4*c14)
)

Hloc = sp.expand(
    Hkin - lam*chi
)

q_aux = [s,v1,v2,v3,lam]
p_aux = [ps,p1,p2,p3,plam]

def PB(F,G):
    return sp.expand(sum(
        sp.diff(F,q)*sp.diff(G,p)
        -
        sp.diff(F,p)*sp.diff(G,q)
        for q,p in zip(q_aux,p_aux)
    ))

assert PB(s,ps) == 1
assert PB(lam,plam) == 1
assert PB(plam,Hloc) == chi

psi = sp.factor(
    PB(chi,Hloc)
)

Delta = sp.factor(
    PB(chi,psi)
)

Akin = sp.factor(
    PB(psi,Hkin)
)

rho = sp.factor(
    PB(psi,Hloc)
)

psi_expected = (
    s*ps/ct + vp/c14
)

Delta_expected = (
    2*(vnorm2/c14 - s**2/ct)
)

Akin_expected = (
    -ps**2/(2*ct**2)
    + pnorm2/(2*c14**2)
)

assert sp.simplify(psi-psi_expected) == 0
assert sp.simplify(Delta-Delta_expected) == 0
assert sp.simplify(Akin-Akin_expected) == 0
assert sp.simplify(rho-(Akin+lam*Delta)) == 0

print("p_lambda -> chi: PASS")
print("chi -> psi: PASS")
print("psi -> rho: PASS")
print("Delta =",Delta)


p_lambda -> chi: PASS
chi -> psi: PASS
psi -> rho: PASS
Delta = -2*(c14*s**2 - c_time*v1**2 - c_time*v2**2 - c_time*v3**2)/(c14*c_time)



## 3 — Base auxiliaire réelle

La base utilisée dans tout ce notebook est exclusivement :

\[
\boxed{
\Phi_0=p_\lambda,
\qquad
\Phi_1=\chi,
\qquad
\Phi_2=\psi,
\qquad
\Phi_3=\rho.
}
\]

Aucune \(\Phi_A\) supplémentaire n'est ajoutée artificiellement.


In [3]:

PHI = [plam, chi, psi, rho]
PHI_NAMES = [
    "p_lambda",
    "chi",
    "psi",
    "rho"
]

AUXILIARY_GVH_BASIS_ACTUAL = True
AUXILIARY_GVH_BASIS_COUNT = len(PHI)

assert AUXILIARY_GVH_BASIS_COUNT == 4

for name,expr in zip(PHI_NAMES,PHI):
    print(name,"=")
    sp.pprint(expr)
    print()


p_lambda =
p_λ

chi =
   2     2     2     2    
- s  + v₁  + v₂  + v₃  + 1

psi =
c₁₄⋅pₛ⋅s + cₜᵢₘₑ⋅p₁⋅v₁ + cₜᵢₘₑ⋅p₂⋅v₂ + cₜᵢₘₑ⋅p₃⋅v₃
──────────────────────────────────────────────────
                    c₁₄⋅cₜᵢₘₑ                     

rho =
       2              2      2   2              2         2              2     ↪
- 4⋅c₁₄ ⋅cₜᵢₘₑ⋅λₘᵤₗₜ⋅s  - c₁₄ ⋅pₛ  + 4⋅c₁₄⋅cₜᵢₘₑ ⋅λₘᵤₗₜ⋅v₁  + 4⋅c₁₄⋅cₜᵢₘₑ ⋅λₘᵤ ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                        2      ↪
                                                                   2⋅c₁₄ ⋅cₜᵢₘ ↪

↪      2              2         2        2   2        2   2        2   2
↪ ₗₜ⋅v₂  + 4⋅c₁₄⋅cₜᵢₘₑ ⋅λₘᵤₗₜ⋅v₃  + cₜᵢₘₑ ⋅p₁  + cₜᵢₘₑ ⋅p₂  + cₜᵢₘₑ ⋅p₃ 
↪ ──────────────────────────────────────────────────────────────────────
↪  2                                                                    
↪ ₑ                                                        


## 4 — Matrice de Poisson auxiliaire réelle

On calcule directement :

\[
C_{AB}
=
\{\Phi_A,\Phi_B\}.
\]

La structure amont attend :

\[
C_4=
\begin{pmatrix}
0&0&0&-\Delta\\
0&0&\Delta&x\\
0&-\Delta&0&y\\
\Delta&-x&-y&0
\end{pmatrix},
\]

où

\[
x=\{\chi,\rho\},
\qquad
y=\{\psi,\rho\}.
\]

Le déterminant doit être :

\[
\boxed{\det C_4=\Delta^4}.
\]


In [4]:

x_aux = sp.factor(PB(chi,rho))
y_aux = sp.factor(PB(psi,rho))

C4_direct = sp.Matrix([
    [sp.factor(PB(PHI[i],PHI[j]))
     for j in range(4)]
    for i in range(4)
])

C4_struct = sp.Matrix([
    [0,      0,       0,       -Delta],
    [0,      0,       Delta,    x_aux],
    [0,     -Delta,   0,        y_aux],
    [Delta, -x_aux,  -y_aux,    0],
])

assert sp.simplify(C4_direct-C4_struct) == sp.zeros(4)
assert C4_direct + C4_direct.T == sp.zeros(4)

det_C4 = sp.factor(C4_direct.det())

assert sp.simplify(det_C4-Delta**4) == 0

AUXILIARY_C4_EXACT = True
AUXILIARY_C4_DET_DELTA4 = True
AUXILIARY_C4_RANK_GENERIC = 4

print("C4 exact structured form: PASS")
print("det(C4) = Delta^4: PASS")
print("generic rank =",AUXILIARY_C4_RANK_GENERIC)


C4 exact structured form: PASS
det(C4) = Delta^4: PASS
generic rank = 4



## 5 — Classification Dirac du secteur auxiliaire

Sur :

\[
\boxed{
c_{14}\neq0,\qquad
c_{\rm time}\neq0,\qquad
\Delta\neq0,
}
\]

on a :

\[
\det C_4\neq0.
\]

Donc les quatre contraintes :

\[
(p_\lambda,\chi,\psi,\rho)
\]

forment une famille de **quatre contraintes de seconde classe** sur cette branche.

Ce point change la stratégie HH :

- une réduction modulo l'idéal des contraintes reste utile pour tester une fermeture faible du crochet canonique ;
- mais la dynamique réduite physiquement doit également connaître le **crochet de Dirac** associé à \(C_4^{-1}\).

Les deux diagnostics seront donc conservés séparément.


In [5]:

AUXILIARY_SECOND_CLASS_GENERIC = bool(
    AUXILIARY_C4_DET_DELTA4
)

assert AUXILIARY_SECOND_CLASS_GENERIC

print(
    "AUXILIARY_SECOND_CLASS_GENERIC =",
    AUXILIARY_SECOND_CLASS_GENERIC
)


AUXILIARY_SECOND_CLASS_GENERIC = True



## 6 — Inverse exact du bloc \(C_4\)

Pour la structure :

\[
C_4=
\begin{pmatrix}
0&0&0&-\Delta\\
0&0&\Delta&x\\
0&-\Delta&0&y\\
\Delta&-x&-y&0
\end{pmatrix},
\]

l'inverse exact est :

\[
\boxed{
C_4^{-1}
=
\begin{pmatrix}
0 & y/\Delta^2 & -x/\Delta^2 & 1/\Delta\\
-y/\Delta^2 & 0 & -1/\Delta & 0\\
x/\Delta^2 & 1/\Delta & 0 & 0\\
-1/\Delta & 0 & 0 & 0
\end{pmatrix}.
}
\]

Il n'existe donc qu'en dehors de la surface :

\[
\Delta=0.
\]


In [6]:

C4_inv = sp.Matrix([
    [0,
     y_aux/Delta**2,
     -x_aux/Delta**2,
     1/Delta],

    [-y_aux/Delta**2,
     0,
     -1/Delta,
     0],

    [x_aux/Delta**2,
     1/Delta,
     0,
     0],

    [-1/Delta,
     0,
     0,
     0],
])

assert sp.simplify(
    C4_direct*C4_inv-sp.eye(4)
) == sp.zeros(4)

assert sp.simplify(
    C4_inv*C4_direct-sp.eye(4)
) == sp.zeros(4)

DIRAC_MATRIX_INVERSE_EXACT = True

print(
    "DIRAC_MATRIX_INVERSE_EXACT =",
    DIRAC_MATRIX_INVERSE_EXACT
)


DIRAC_MATRIX_INVERSE_EXACT = True



## 7 — Noyau de correction de Dirac

Pour deux fonctionnelles \(F,G\) :

\[
\{F,G\}_D
=
\{F,G\}
-
\{F,\Phi_A\}
(C^{-1})^{AB}
\{\Phi_B,G\}.
\]

Si :

\[
a_A(F)=\{F,\Phi_A\},
\qquad
a_A(G)=\{G,\Phi_A\},
\]

alors :

\[
\{\Phi_B,G\}
=
-a_B(G),
\]

d'où :

\[
\boxed{
\{F,G\}_D
=
\{F,G\}
+
a_A(F)(C^{-1})^{AB}a_B(G).
}
\]

Cette correction est antisymétrique.

Pour le futur HH :

\[
F=H[N],
\qquad
G=H[M].
\]

Cette formule est **l'interface exacte** que `.1.6` devra évaluer.


In [7]:

aN = sp.Matrix(
    sp.symbols("aN0:4")
)
aM = sp.Matrix(
    sp.symbols("aM0:4")
)

dirac_correction_template = sp.factor(
    (aN.T*C4_inv*aM)[0]
)

swap_template = sp.factor(
    (aM.T*C4_inv*aN)[0]
)

assert sp.simplify(
    dirac_correction_template
    +
    swap_template
) == 0

DIRAC_CORRECTION_TEMPLATE_EXPLICIT = True
DIRAC_CORRECTION_ANTISYMMETRIC = True

print("Dirac correction template:")
sp.pprint(dirac_correction_template)

print(
    "DIRAC_CORRECTION_ANTISYMMETRIC =",
    DIRAC_CORRECTION_ANTISYMMETRIC
)


Dirac correction template:
 ⎛             3              2              3   2                      3      ↪
-⎝4⋅aM₀⋅aN₁⋅c₁₄ ⋅cₜᵢₘₑ⋅λₘᵤₗₜ⋅s  - aM₀⋅aN₁⋅c₁₄ ⋅pₛ  - 4⋅aM₀⋅aN₁⋅c₁₄⋅cₜᵢₘₑ ⋅λₘᵤₗ ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪

↪     2                      3         2                      3         2      ↪
↪ ₜ⋅v₁  - 4⋅aM₀⋅aN₁⋅c₁₄⋅cₜᵢₘₑ ⋅λₘᵤₗₜ⋅v₂  - 4⋅aM₀⋅aN₁⋅c₁₄⋅cₜᵢₘₑ ⋅λₘᵤₗₜ⋅v₃  + aM ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪



## 8 — Vérification forte : les contraintes ont crochet de Dirac nul

La propriété requise est :

\[
\boxed{
\{\Phi_A,F\}_D=0
}
\]

pour toute fonctionnelle \(F\), sur la branche où \(C_4^{-1}\) existe.

Le contrôle est fait abstraitement à partir de :

\[
C_{AB}(C^{-1})^{BC}=\delta_A{}^C.
\]


In [8]:

zF = sp.Matrix(
    sp.symbols("zF0:4")
)

# zF[B] = {Phi_B,F}
# Dirac bracket:
# {Phi_A,F}_D = zF[A] - C_AB C^{-1 BC} zF[C]
dirac_constraint_residual = sp.simplify(
    zF - C4_direct*C4_inv*zF
)

assert dirac_constraint_residual == sp.zeros(4,1)

DIRAC_CONSTRAINTS_STRONGLY_ZERO = True

print(
    "DIRAC_CONSTRAINTS_STRONGLY_ZERO =",
    DIRAC_CONSTRAINTS_STRONGLY_ZERO
)


DIRAC_CONSTRAINTS_STRONGLY_ZERO = True



# Partie II — Idéal polynomial réel des contraintes

## 9 — Polynomialisation sans changer la branche générique

Les contraintes \(\psi\) et \(\rho\) comportent des dénominateurs en
\(c_{14}\) et \(c_{\rm time}\).

Sur :

\[
c_{14}\neq0,
\qquad
c_{\rm time}\neq0,
\]

on peut multiplier par des facteurs non nuls et employer :

\[
\Phi_0^{\rm poly}=p_\lambda,
\]

\[
\Phi_1^{\rm poly}=\chi,
\]

\[
\boxed{
\Phi_2^{\rm poly}
=
c_{14}c_{\rm time}\psi
=
c_{14}s p_s
+
c_{\rm time}v^ip_i,
}
\]

et :

\[
\boxed{
\Phi_3^{\rm poly}
=
2c_{14}^2c_{\rm time}^2\rho.
}
\]

Ce sont les contraintes réellement dérivées, avec seulement leurs dénominateurs éliminés.


In [9]:

Phi0_poly = plam

Phi1_poly = sp.expand(
    chi
)

Phi2_poly = sp.expand(
    c14*ct*psi
)

Phi3_poly = sp.expand(
    2*c14**2*ct**2*rho
)

assert sp.simplify(
    Phi2_poly
    -
    (
        c14*s*ps
        +
        ct*vp
    )
) == 0

expected_phi3 = sp.expand(
    -c14**2*ps**2
    +
    ct**2*pnorm2
    +
    4*c14*ct*lam*(
        ct*vnorm2
        -
        c14*s**2
    )
)

assert sp.simplify(
    Phi3_poly-expected_phi3
) == 0

PHI_POLY = [
    Phi0_poly,
    Phi1_poly,
    Phi2_poly,
    Phi3_poly
]

POLYNOMIALIZATION_EXACT_ON_GENERIC_COUPLING_BRANCH = True

print("Polynomialized actual constraints: PASS")


Polynomialized actual constraints: PASS



## 10 — Base de Gröbner dans le vrai anneau polynomial

Pour éviter d'inverser implicitement des combinaisons non autorisées telles que
\(c_{14}-c_{\rm time}\), les couplages sont gardés comme **variables polynomiales**.

On travaille sur :

\[
\mathbb Q[
p_\lambda,\lambda,p_s,p_i,s,v_i,c_{14},c_{\rm time}
].
\]

Le test :

\[
R\in
\langle
\Phi_0^{\rm poly},
\Phi_1^{\rm poly},
\Phi_2^{\rm poly},
\Phi_3^{\rm poly}
\rangle
\]

est alors un test polynomial strict.

Une non-appartenance à cet idéal polynomial n'exclut pas encore une annulation spécifique à la branche localisée
\(c_{14}c_{\rm time}\Delta\neq0\).
C'est pourquoi un deuxième moteur sera construit ensuite.


In [10]:

ideal_variables = [
    plam,
    lam,
    ps,
    p1,p2,p3,
    s,
    v1,v2,v3,
    c14,ct
]

GB_actual = sp.groebner(
    PHI_POLY,
    *ideal_variables,
    order="grevlex",
    domain=sp.QQ
)

ACTUAL_GVH_GROEBNER_BASIS_BUILT = True
ACTUAL_GVH_GROEBNER_SIZE = len(
    GB_actual.polys
)

print(
    "ACTUAL_GVH_GROEBNER_SIZE =",
    ACTUAL_GVH_GROEBNER_SIZE
)

assert ACTUAL_GVH_GROEBNER_SIZE >= 4


ACTUAL_GVH_GROEBNER_SIZE = 23



## 11 — Régression positive sur la vraie base

Ce contrôle n'est **pas** un résidu GVH physique.

Il vérifie seulement que le moteur reconnaît correctement une combinaison construite à partir des **vraies** contraintes :

\[
R_{\rm control}
=
f^A\Phi_A^{GVH}.
\]

Un résidu voisin non construit dans l'idéal doit rester non nul.


In [11]:

R_control_member = sp.expand(
    (s+p1)*Phi0_poly
    +
    (p2+v1)*Phi1_poly
    +
    (lam+s)*Phi2_poly
    +
    (v3+1)*Phi3_poly
)

R_control_nonmember = sp.expand(
    s + v1 + p2 + 1
)

member_remainder = sp.expand(
    GB_actual.reduce(
        R_control_member
    )[1]
)

nonmember_remainder = sp.expand(
    GB_actual.reduce(
        R_control_nonmember
    )[1]
)

assert member_remainder == 0
assert nonmember_remainder != 0

ACTUAL_IDEAL_MEMBER_CONTROL_PASS = True
ACTUAL_IDEAL_NONMEMBER_CONTROL_PASS = True

print(
    "member control remainder =",
    member_remainder
)
print(
    "nonmember control remainder =",
    nonmember_remainder
)


member control remainder = 0
nonmember control remainder = p2 + s + v1 + 1



# Partie III — Réduction localisée sur la branche physique générique

## 12 — Pourquoi une deuxième réduction est nécessaire

L'idéal polynomial strict ne tient pas compte des inversions autorisées par :

\[
c_{14}\neq0,
\qquad
c_{\rm time}\neq0,
\qquad
\Delta\neq0.
\]

De plus, sur la contrainte de norme réelle :

\[
\chi=0
\quad\Longrightarrow\quad
s^2=1+v^2,
\]

donc :

\[
\boxed{s\neq0}
\]

sur la branche timelike réelle.

On peut donc résoudre sans choisir arbitrairement une composante \(v_i\) non nulle :

\[
\psi=0
\quad\Longrightarrow\quad
\boxed{
p_s
=
-\frac{c_{\rm time}}{c_{14}s}
v^ip_i
}
\]

puis :

\[
\rho=0
\quad\Longrightarrow\quad
\boxed{
\lambda_{\rm mult}
=
-\frac{\mathcal A}{\Delta}.
}
\]

Cette procédure définit une réduction **localisée sur la branche générique**, complémentaire de Gröbner.


In [12]:

ps_on_aux = sp.factor(
    -ct*vp/(c14*s)
)

A_on_aux = sp.factor(
    Akin.subs(
        ps,
        ps_on_aux
    )
)

Delta_on_aux = sp.factor(
    Delta
)

lambda_on_aux = sp.factor(
    -A_on_aux/Delta_on_aux
)

assert sp.simplify(
    psi.subs(ps,ps_on_aux)
) == 0

assert sp.simplify(
    rho
    .subs(ps,ps_on_aux)
    .subs(lam,lambda_on_aux)
) == 0

print("p_s on auxiliary surface =")
sp.pprint(ps_on_aux)

print("\nlambda on auxiliary surface =")
sp.pprint(lambda_on_aux)


p_s on auxiliary surface =
-cₜᵢₘₑ⋅(p₁⋅v₁ + p₂⋅v₂ + p₃⋅v₃) 
───────────────────────────────
             c₁₄⋅s             

lambda on auxiliary surface =
      ⎛  2  2     2   2                                     2  2     2   2     ↪
cₜᵢₘₑ⋅⎝p₁ ⋅s  - p₁ ⋅v₁  - 2⋅p₁⋅p₂⋅v₁⋅v₂ - 2⋅p₁⋅p₃⋅v₁⋅v₃ + p₂ ⋅s  - p₂ ⋅v₂  - 2 ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                   2 ⎛     2           2           2           ↪
                            4⋅c₁₄⋅s ⋅⎝c₁₄⋅s  - cₜᵢₘₑ⋅v₁  - cₜᵢₘₑ⋅v₂  - cₜᵢₘₑ⋅v ↪

↪                  2  2     2   2⎞
↪ ⋅p₂⋅p₃⋅v₂⋅v₃ + p₃ ⋅s  - p₃ ⋅v₃ ⎠
↪ ────────────────────────────────
↪  2⎞                             
↪ ₃ ⎠                             



## 13 — Normal-form localisée

Après :

\[
p_\lambda=0,
\qquad
p_s=p_s^{(\Phi)},
\qquad
\lambda=\lambda^{(\Phi)},
\]

le dernier quotient est :

\[
s^2-(1+v^2)=0.
\]

Pour déterminer si un résidu rationnel est nul sur la branche auxiliaire, il suffit de réduire son numérateur modulo ce polynôme quadratique en \(s\).

La fonction suivante retourne :

- le reste du numérateur ;
- le dénominateur après réduction.

Le résultat est **weak-zero on generic branch** seulement si le reste du numérateur est exactement nul.


In [13]:

norm_poly_s = sp.Poly(
    s**2-(1+vnorm2),
    s
)

def localized_auxiliary_normal_form(expr):
    e = sp.together(expr)

    e = sp.together(
        e.subs(plam,0)
    )

    e = sp.together(
        e.subs(
            ps,
            ps_on_aux
        )
    )

    e = sp.together(
        e.subs(
            lam,
            lambda_on_aux
        )
    )

    e = sp.cancel(e)

    num,den = sp.fraction(e)

    num = sp.expand(num)
    den = sp.factor(den)

    rem_num = sp.rem(
        sp.Poly(num,s),
        norm_poly_s
    ).as_expr()

    rem_num = sp.factor(
        sp.expand(rem_num)
    )

    return rem_num, den

for name,phi in zip(PHI_NAMES,PHI):
    rem,den = localized_auxiliary_normal_form(phi)
    print(name,"localized remainder =",rem)
    assert rem == 0

LOCALIZED_ACTUAL_AUXILIARY_REDUCTION_PASS = True


p_lambda localized remainder = 0
chi localized remainder = 0
psi localized remainder = 0
rho localized remainder = 0



## 14 — Contrôles de la réduction localisée


In [14]:

loc_member_rem,loc_member_den = (
    localized_auxiliary_normal_form(
        R_control_member
    )
)

loc_nonmember_rem,loc_nonmember_den = (
    localized_auxiliary_normal_form(
        R_control_nonmember
    )
)

assert loc_member_rem == 0
assert loc_nonmember_rem != 0

LOCALIZED_MEMBER_CONTROL_PASS = True
LOCALIZED_NONMEMBER_CONTROL_PASS = True

print(
    "localized member remainder =",
    loc_member_rem
)

print(
    "localized nonmember remainder =",
    loc_nonmember_rem
)


localized member remainder = 0
localized nonmember remainder = p2 + s + v1 + 1



# Partie IV — Interface du vrai résidu auxiliaire

## 15 — Règle de non-fabrication

Les notebooks amont ont défini :

\[
R_{HH}
=
\{H[N],H[M]\}_{\rm can}
-
D[\beta],
\]

mais ils n'ont pas encore matérialisé le crochet HH full-field.

Par conséquent, ils ne fournissent pas encore un objet :

\[
R_{\rm aux}^{GVH}[N,M]
\]

séparable et testable.

Cette étape interdit donc :

```text
R_aux_GVH = polynomial invented by hand
```

Le seul objet physique acceptable sera celui extrait de `.1.6` après calcul effectif de :

\[
\{H[N],H[M]\}.
\]


In [15]:

R_AUX_GVH_MATERIALIZED_UPSTREAM = False
R_AUX_GVH_REDUCED = False

NON_FABRICATION_GATE = (
    not R_AUX_GVH_MATERIALIZED_UPSTREAM
    and not R_AUX_GVH_REDUCED
)

assert NON_FABRICATION_GATE

print(
    "R_AUX_GVH_MATERIALIZED_UPSTREAM =",
    R_AUX_GVH_MATERIALIZED_UPSTREAM
)

print(
    "R_AUX_GVH_REDUCED =",
    R_AUX_GVH_REDUCED
)

print(
    "NON_FABRICATION_GATE =",
    NON_FABRICATION_GATE
)


R_AUX_GVH_MATERIALIZED_UPSTREAM = False
R_AUX_GVH_REDUCED = False
NON_FABRICATION_GATE = True



## 16 — Classifieur prêt pour `.1.6`

Une fois un résidu concret fourni :

\[
R_{\rm test},
\]

la classification devra respecter cet ordre :

### A. fermeture forte

\[
R_{\rm test}=0.
\]

### B. fermeture faible — idéal polynomial

\[
R_{\rm test}
\in
\langle\Phi_A^{\rm poly}\rangle.
\]

### C. fermeture faible — branche localisée

Si le reste polynomial est non nul mais :

\[
R_{\rm test}
\approx0
\]

après réduction sur :

\[
c_{14}c_{\rm time}\Delta s\neq0,
\]

alors la fermeture est faible **uniquement sur cette branche générique**.

### D. résidu irréductible

Si les deux réductions donnent un reste non nul :

\[
R_{\rm test}\not\approx0.
\]

Ce classement ne sera appliqué à \(R_{HH}\) qu'après sa matérialisation réelle.


In [16]:

def classify_auxiliary_residual(expr):
    expr0 = sp.factor(
        sp.expand(expr)
    )

    if expr0 == 0:
        return {
            "status":
                "STRONG_ZERO",
            "polynomial_remainder":
                sp.Integer(0),
            "localized_remainder":
                sp.Integer(0),
        }

    poly_rem = None

    try:
        poly_rem = sp.expand(
            GB_actual.reduce(
                sp.expand(expr0)
            )[1]
        )
    except Exception:
        poly_rem = sp.Symbol(
            "NOT_POLYNOMIAL"
        )

    if poly_rem == 0:
        return {
            "status":
                "WEAK_ZERO_POLYNOMIAL_IDEAL",
            "polynomial_remainder":
                sp.Integer(0),
            "localized_remainder":
                sp.Integer(0),
        }

    loc_rem,loc_den = (
        localized_auxiliary_normal_form(
            expr0
        )
    )

    if loc_rem == 0:
        return {
            "status":
                "WEAK_ZERO_GENERIC_LOCALIZED_BRANCH",
            "polynomial_remainder":
                poly_rem,
            "localized_remainder":
                sp.Integer(0),
            "localized_denominator":
                loc_den,
        }

    return {
        "status":
            "IRREDUCIBLE_ON_TESTED_GENERIC_BRANCH",
        "polynomial_remainder":
            poly_rem,
        "localized_remainder":
            loc_rem,
        "localized_denominator":
            loc_den,
    }

assert (
    classify_auxiliary_residual(
        0
    )["status"]
    ==
    "STRONG_ZERO"
)

assert (
    classify_auxiliary_residual(
        R_control_member
    )["status"]
    ==
    "WEAK_ZERO_POLYNOMIAL_IDEAL"
)

assert (
    classify_auxiliary_residual(
        R_control_nonmember
    )["status"]
    ==
    "IRREDUCIBLE_ON_TESTED_GENERIC_BRANCH"
)

AUXILIARY_RESIDUAL_CLASSIFIER_READY = True

print(
    "AUXILIARY_RESIDUAL_CLASSIFIER_READY =",
    AUXILIARY_RESIDUAL_CLASSIFIER_READY
)


AUXILIARY_RESIDUAL_CLASSIFIER_READY = True



## 17 — Gate critique pour le HH final

Parce que :

\[
\operatorname{rank}C_4=4,
\]

le secteur auxiliaire est second-class sur la branche générique.

Le notebook `.1.6` devra donc distinguer **deux questions** :

### Question 1 — Crochet canonique

\[
R_{HH}^{\rm can}
=
\{H[N],H[M]\}_{\rm can}
-
D[\beta].
\]

Est-il :

\[
0,\qquad
\approx0\pmod{\Phi_A},
\qquad
\text{ou irréductible}?
\]

### Question 2 — Crochet de Dirac réduit

\[
R_{HH}^{D}
=
\{H[N],H[M]\}_{D}
-
D[\beta].
\]

La classification physique de l'algèbre réduite ne doit pas confondre ces deux objets.

Donc le simple test :

\[
R_{HH}^{\rm can}\in\langle\Phi_A\rangle
\]

est nécessaire pour une fermeture faible canonique, mais **ne remplace pas** l'audit du crochet de Dirac lorsque les contraintes auxiliaires sont de seconde classe.


In [17]:

CANONICAL_HH_RESIDUAL_REQUIRED = True
DIRAC_HH_RESIDUAL_REQUIRED = True

CANONICAL_VS_DIRAC_DISTINCTION_REGISTERED = True

assert CANONICAL_HH_RESIDUAL_REQUIRED
assert DIRAC_HH_RESIDUAL_REQUIRED

print(
    "CANONICAL_VS_DIRAC_DISTINCTION_REGISTERED =",
    CANONICAL_VS_DIRAC_DISTINCTION_REGISTERED
)


CANONICAL_VS_DIRAC_DISTINCTION_REGISTERED = True



## 18 — Table des gates


In [18]:

GATES = {
    "actual_GVH_auxiliary_basis_imported":
        AUXILIARY_GVH_BASIS_ACTUAL,

    "actual_GVH_auxiliary_basis_count_4":
        AUXILIARY_GVH_BASIS_COUNT == 4,

    "constraint_chain_rederived":
        True,

    "C4_exact":
        AUXILIARY_C4_EXACT,

    "C4_det_Delta4":
        AUXILIARY_C4_DET_DELTA4,

    "C4_generic_rank4":
        AUXILIARY_C4_RANK_GENERIC == 4,

    "auxiliary_second_class_generic":
        AUXILIARY_SECOND_CLASS_GENERIC,

    "Dirac_matrix_inverse_exact":
        DIRAC_MATRIX_INVERSE_EXACT,

    "Dirac_correction_template_explicit":
        DIRAC_CORRECTION_TEMPLATE_EXPLICIT,

    "Dirac_constraints_strongly_zero":
        DIRAC_CONSTRAINTS_STRONGLY_ZERO,

    "polynomialization_exact":
        POLYNOMIALIZATION_EXACT_ON_GENERIC_COUPLING_BRANCH,

    "actual_GVH_Groebner_basis_built":
        ACTUAL_GVH_GROEBNER_BASIS_BUILT,

    "actual_ideal_member_control":
        ACTUAL_IDEAL_MEMBER_CONTROL_PASS,

    "actual_ideal_nonmember_control":
        ACTUAL_IDEAL_NONMEMBER_CONTROL_PASS,

    "localized_actual_auxiliary_reduction":
        LOCALIZED_ACTUAL_AUXILIARY_REDUCTION_PASS,

    "localized_member_control":
        LOCALIZED_MEMBER_CONTROL_PASS,

    "localized_nonmember_control":
        LOCALIZED_NONMEMBER_CONTROL_PASS,

    "auxiliary_residual_classifier_ready":
        AUXILIARY_RESIDUAL_CLASSIFIER_READY,

    "canonical_vs_Dirac_distinction_registered":
        CANONICAL_VS_DIRAC_DISTINCTION_REGISTERED,

    "R_aux_GVH_materialized_upstream":
        R_AUX_GVH_MATERIALIZED_UPSTREAM,

    "R_aux_GVH_reduced":
        R_AUX_GVH_REDUCED,

    "full_HH_canonical_bracket_computed":
        False,

    "full_HH_Dirac_bracket_computed":
        False,

    "RHH_physical_classified":
        False,

    "hypersurface_algebra_closed":
        False,
}

for k,v in GATES.items():
    print(k,":",v)


actual_GVH_auxiliary_basis_imported : True
actual_GVH_auxiliary_basis_count_4 : True
constraint_chain_rederived : True
C4_exact : True
C4_det_Delta4 : True
C4_generic_rank4 : True
auxiliary_second_class_generic : True
Dirac_matrix_inverse_exact : True
Dirac_correction_template_explicit : True
Dirac_constraints_strongly_zero : True
polynomialization_exact : True
actual_GVH_Groebner_basis_built : True
actual_ideal_member_control : True
actual_ideal_nonmember_control : True
localized_actual_auxiliary_reduction : True
localized_member_control : True
localized_nonmember_control : True
auxiliary_residual_classifier_ready : True
canonical_vs_Dirac_distinction_registered : True
R_aux_GVH_materialized_upstream : False
R_aux_GVH_reduced : False
full_HH_canonical_bracket_computed : False
full_HH_Dirac_bracket_computed : False
RHH_physical_classified : False
hypersurface_algebra_closed : False



## 19 — Verdict autorisé de `.1.5`

Le résultat attendu si tous les contrôles exécutables passent est :

\[
\boxed{
\texttt{
PASS-ACTUAL-GVH-AUXILIARY-BASIS-
C4-RANK4-DIRAC-AND-CONSTRAINT-IDEAL-REDUCTION-READY
}
}
\]

avec la réserve méthodologique obligatoire :

\[
\boxed{
\texttt{
R-AUX-GVH-DEFERRED-UNTIL-ACTUAL-HH-MATERIALIZATION
}
}
\]

Ce n'est **pas un échec** de `.1.5`.

C'est la résolution correcte d'une dépendance logique :

\[
R_{\rm aux}^{GVH}
\subset
R_{HH}
\]

ne peut être identifié avant que \(R_{HH}\) existe.

Après `.1.5`, tous les outils auxiliaires sont prêts pour `.1.6`.


In [19]:

REQUIRED_READY = [
    "actual_GVH_auxiliary_basis_imported",
    "actual_GVH_auxiliary_basis_count_4",
    "constraint_chain_rederived",
    "C4_exact",
    "C4_det_Delta4",
    "C4_generic_rank4",
    "auxiliary_second_class_generic",
    "Dirac_matrix_inverse_exact",
    "Dirac_correction_template_explicit",
    "Dirac_constraints_strongly_zero",
    "polynomialization_exact",
    "actual_GVH_Groebner_basis_built",
    "actual_ideal_member_control",
    "actual_ideal_nonmember_control",
    "localized_actual_auxiliary_reduction",
    "localized_member_control",
    "localized_nonmember_control",
    "auxiliary_residual_classifier_ready",
    "canonical_vs_Dirac_distinction_registered",
]

ALL_AUXILIARY_TOOLS_READY = all(
    GATES[k]
    for k in REQUIRED_READY
)

if ALL_AUXILIARY_TOOLS_READY:
    FINAL_STATUS = (
        "PASS-ACTUAL-GVH-AUXILIARY-BASIS-"
        "C4-RANK4-DIRAC-AND-CONSTRAINT-IDEAL-REDUCTION-READY_"
        "R-AUX-GVH-DEFERRED-UNTIL-ACTUAL-HH-MATERIALIZATION"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-AUXILIARY-BASIS-OR-REDUCTION-ENGINE-FAIL"
    )

R_HH_STATUS = "BLOCKED-PENDING-.1.6"
DISPERSION_READY = False

print("FINAL_STATUS =",FINAL_STATUS)
print("R_HH_STATUS =",R_HH_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


FINAL_STATUS = PASS-ACTUAL-GVH-AUXILIARY-BASIS-C4-RANK4-DIRAC-AND-CONSTRAINT-IDEAL-REDUCTION-READY_R-AUX-GVH-DEFERRED-UNTIL-ACTUAL-HH-MATERIALIZATION
R_HH_STATUS = BLOCKED-PENDING-.1.6
DISPERSION_READY = False



# 20 — Contrat d'entrée pour `.1.6`

Le notebook `.1.6` devra importer/reconstruire :

\[
H[N]
=
\int d^3x\,N\mathscr C_N
\]

dans une représentation fonctionnelle exploitable, puis calculer :

\[
\boxed{
R_{HH}^{\rm can}
=
\{H[N],H[M]\}_{\rm can}
-
D[\beta]
}
\]

avec :

\[
\beta^i
=
h^{ij}
(ND_jM-MD_jN).
\]

Ensuite :

1. appliquer `classify_auxiliary_residual` au résidu canonique ;
2. isoler les termes auxiliaires réellement présents ;
3. évaluer :
   \[
   \{H[N],\Phi_A\},
   \qquad
   \{H[M],\Phi_A\};
   \]
4. former la correction exacte :
   \[
   a_A(N)(C^{-1})^{AB}a_B(M);
   \]
5. calculer :
   \[
   R_{HH}^{D};
   \]
6. publier séparément :
   - classification canonique ;
   - classification Dirac réduite.

Seulement alors une conclusion sur la fermeture des hypersurfaces sera autorisée.


In [20]:

artifact = {
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.5",

    "traceability":
        "ACTUAL_UPSTREAM_GVH_CONSTRAINTS_NON_FABRICATION",

    "generic_branch":
        [
            "c14 != 0",
            "c_time != 0",
            "Delta != 0",
        ],

    "actual_auxiliary_basis":
        {
            "Phi0":"p_lambda",
            "Phi1":"chi=-s^2+v^2+1",
            "Phi2":"psi=s*p_s/c_time + v^i*p_i/c14",
            "Phi3":"rho=Akin+lambda_mult*Delta",
        },

    "C4":
        {
            "det":"Delta^4",
            "rank_generic":4,
            "classification":"SECOND_CLASS",
        },

    "Dirac_inverse":
        "EXPLICIT",

    "constraint_reduction":
        {
            "polynomial_Groebner":
                "READY",
            "generic_localized":
                "READY",
            "classifier":
                "READY",
        },

    "R_aux_GVH_materialized_upstream":
        False,

    "R_aux_GVH_reduced":
        False,

    "reason_R_aux_deferred":
        (
            "Actual auxiliary HH residual is an output of the "
            "full HH computation and is not materially present upstream."
        ),

    "canonical_HH_required":
        True,

    "Dirac_HH_required":
        True,

    "final_status":
        FINAL_STATUS,

    "RHH_status":
        R_HH_STATUS,

    "hypersurface_algebra_closed":
        False,

    "dispersion_ready":
        False,

    "next":
        (
            "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6_"
            "Strict_Full_Field_Canonical_and_Dirac_HH_Residual_Classification.ipynb"
        ),
}

export_dir = (
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path.cwd()/"gvh_exports"
)

export_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.1.5_"
    "auxiliary_constraint_ideal_and_dirac_ready.json"
)

artifact_path.write_text(
    json.dumps(
        artifact,
        indent=2
    ),
    encoding="utf-8"
)

print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.1.5_auxiliary_constraint_ideal_and_dirac_ready.json



# Conclusion

Cette étape utilise la **vraie chaîne auxiliaire GVH** :

\[
\boxed{
\Phi_A
=
(p_\lambda,\chi,\psi,\rho)
}
\]

et redérive :

\[
\boxed{
\det C_4=\Delta^4,
\qquad
\operatorname{rank}C_4=4
}
\]

sur la branche générique.

Elle prépare :

- la réduction de Gröbner sur la vraie base ;
- la réduction localisée sur
  \[
  c_{14}c_{\rm time}\Delta s\neq0;
  \]
- le classifieur de résidu ;
- l'inverse exact \(C_4^{-1}\) ;
- le noyau de correction du crochet de Dirac.

Mais elle **n'invente pas** un \(R_{\rm aux}^{GVH}\) inexistant en amont.

Le vrai \(R_{\rm aux}^{GVH}\) sera extrait de :

\[
R_{HH}^{\rm can}
\]

dans `.1.6`, puis réduit par les outils préparés ici.

Ainsi la prochaine étape peut être :

\[
\boxed{
\mathbf{0.3.2.7.3.7.3.3.1.6}
}
\]

avec toujours :

\[
\boxed{
R_{HH}
=
\texttt{BLOCKED-PENDING-.1.6},
\qquad
\mathrm{DISPERSION\_READY=False}.
}
\]
